In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 72 — Workflow-Driven Data Agent (LlamaIndex)

## Goal
Use LlamaIndex **Workflows** for multi-step data tasks
with RAG — an event-driven orchestration layer.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **Workflow** | Event-driven orchestration in LlamaIndex |
| **Steps** | Decorated functions that handle events |
| **Custom events** | Route data between steps |
| **RAG integration** | Combine retrieval + generation in a workflow |
| **Multi-step reasoning** | Chain steps: query → retrieve → analyze → respond |

## Stack
- `llama-index-core` 0.14+ (Workflow API)
- `llama-index-llms-ollama`
- `llama-index-embeddings-huggingface`

In [ ]:
import os, warnings, json
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.core.workflow import (
    Workflow, StartEvent, StopEvent, step, Event,
)
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

print("All imports OK")

In [ ]:
# Configure LLM & Embeddings
Settings.llm = Ollama(model="qwen3.5:9b", request_timeout=120.0, temperature=0)
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
Settings.chunk_size = 256
Settings.chunk_overlap = 50

print(f"LLM: {Settings.llm.model}")
print(f"Embed: {Settings.embed_model.model_name}")

## Step 1 — Create Sample Knowledge Base

We build a small document collection about different
programming paradigms and index it for retrieval.

In [ ]:
documents = [
    Document(text="""Functional Programming (FP)
FP treats computation as evaluation of mathematical functions. Key concepts:
pure functions (no side effects), immutability, first-class functions,
higher-order functions, recursion, pattern matching. Languages: Haskell,
Erlang, Clojure, F#. Benefits: easier testing, parallelism, fewer bugs.
Drawbacks: steeper learning curve, can be less intuitive for stateful tasks.""",
             metadata={"topic": "functional_programming"}),
    
    Document(text="""Object-Oriented Programming (OOP)
OOP organizes code into objects that contain data and behavior. Four pillars:
encapsulation, inheritance, polymorphism, abstraction. Design patterns: Singleton,
Factory, Observer, Strategy. Languages: Java, C++, Python, C#.
Benefits: code reuse, modularity, intuitive modeling. Drawbacks: can lead to
complex hierarchies, tight coupling if misused.""",
             metadata={"topic": "oop"}),
    
    Document(text="""Reactive Programming
Reactive programming deals with asynchronous data streams. Core concepts:
Observables, Observers, Operators, Schedulers. Libraries: RxJS, RxPython,
Project Reactor. Benefits: handles complex async flows elegantly, backpressure
management. Used in: real-time UIs, event-driven microservices, IoT data.""",
             metadata={"topic": "reactive"}),
    
    Document(text="""Logic Programming
Logic programming uses formal logic to express computation. Programs consist
of facts and rules. The engine uses unification and backtracking to derive
answers. Primary language: Prolog. Used in: AI, natural language processing,
database queries, expert systems. Benefits: declarative, good for rule-based
systems. Drawbacks: limited control flow, performance concerns.""",
             metadata={"topic": "logic"}),
    
    Document(text="""Concurrent and Parallel Programming
Concurrency: managing multiple tasks that can overlap in time.
Parallelism: actually executing multiple tasks simultaneously on multiple cores.
Models: threads, async/await, actors (Akka/Erlang), CSP (Go channels).
Challenges: race conditions, deadlocks, data synchronization.
Solutions: locks, semaphores, message passing, STM (Software Transactional Memory).""",
             metadata={"topic": "concurrency"}),
]

# Build index
index = VectorStoreIndex.from_documents(documents)
retriever = index.as_retriever(similarity_top_k=2)

print(f"Indexed {len(documents)} documents about programming paradigms")

## Step 2 — Define Custom Events

Workflows are event-driven. Each step consumes one
event type and emits another, forming a pipeline.

```
StartEvent → QueryEvent → RetrieveEvent → AnalyzeEvent → StopEvent
```

In [ ]:
class QueryEvent(Event):
    """Carries the parsed user query."""
    query: str
    task_type: str  # 'compare', 'explain', 'recommend'

class RetrieveEvent(Event):
    """Carries retrieved context chunks."""
    query: str
    task_type: str
    context: str
    sources: list

class AnalyzeEvent(Event):
    """Carries the LLM's analysis."""
    analysis: str
    sources: list
    task_type: str

print("Custom events defined: QueryEvent → RetrieveEvent → AnalyzeEvent")

## Step 3 — Build a Multi-Step Workflow

The workflow chains four steps:
1. **Parse** the user request to determine task type
2. **Retrieve** relevant documents
3. **Analyze** with the LLM
4. **Format** the final response

In [ ]:
class DataAgentWorkflow(Workflow):
    
    @step
    async def parse_query(self, ev: StartEvent) -> QueryEvent:
        """Step 1: Parse the user query and classify the task."""
        query = ev.query
        query_lower = query.lower()
        
        if any(w in query_lower for w in ["compare", "difference", "vs", "versus"]):
            task_type = "compare"
        elif any(w in query_lower for w in ["recommend", "suggest", "which", "best"]):
            task_type = "recommend"
        else:
            task_type = "explain"
        
        print(f"  📝 Step 1 — Parsed query: task_type='{task_type}'")
        return QueryEvent(query=query, task_type=task_type)
    
    @step
    async def retrieve_context(self, ev: QueryEvent) -> RetrieveEvent:
        """Step 2: Retrieve relevant documents from the index."""
        nodes = retriever.retrieve(ev.query)
        
        context_parts = []
        sources = []
        for node in nodes:
            context_parts.append(node.text)
            sources.append(node.metadata.get("topic", "unknown"))
        
        context = "\n---\n".join(context_parts)
        print(f"  🔍 Step 2 — Retrieved {len(nodes)} chunks from: {sources}")
        
        return RetrieveEvent(
            query=ev.query, task_type=ev.task_type,
            context=context, sources=sources,
        )
    
    @step
    async def analyze(self, ev: RetrieveEvent) -> AnalyzeEvent:
        """Step 3: Use the LLM to analyze retrieved context."""
        task_prompts = {
            "compare": "Compare and contrast the following programming paradigms. "
                       "Highlight key differences, trade-offs, and when to use each.",
            "explain": "Explain the following programming concepts clearly. "
                       "Include key ideas, benefits, and practical applications.",
            "recommend": "Based on the following paradigms, recommend which one "
                         "is best for the user's scenario. Justify your recommendation.",
        }
        
        prompt = f"""{task_prompts[ev.task_type]}

Context:
{ev.context}

User question: {ev.query}

Provide a clear, structured answer."""
        
        llm = Settings.llm
        response = await llm.acomplete(prompt)
        
        print(f"  🤖 Step 3 — LLM analysis complete ({len(str(response))} chars)")
        return AnalyzeEvent(
            analysis=str(response), sources=ev.sources, task_type=ev.task_type,
        )
    
    @step
    async def format_response(self, ev: AnalyzeEvent) -> StopEvent:
        """Step 4: Format the final response with metadata."""
        result = {
            "answer": ev.analysis,
            "task_type": ev.task_type,
            "sources": ev.sources,
        }
        print(f"  ✅ Step 4 — Response formatted")
        return StopEvent(result=result)

workflow = DataAgentWorkflow(timeout=120)
print("Workflow created with 4 steps")

In [ ]:
# Test 1: Comparison query
import asyncio

async def ask(question: str):
    print(f"\n{'='*70}")
    print(f"Question: {question}")
    print(f"{'='*70}")
    result = await workflow.run(query=question)
    print(f"\nTask type: {result['task_type']}")
    print(f"Sources: {result['sources']}")
    print(f"\nAnswer:\n{result['answer'][:800]}")
    return result

result1 = await ask("Compare functional programming vs OOP")

In [ ]:
# Test 2: Explanation query
result2 = await ask("Explain how reactive programming handles async data")

In [ ]:
# Test 3: Recommendation query
result3 = await ask("Which paradigm would you recommend for building a real-time chat app?")

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Workflow** | `class MyWorkflow(Workflow)` — container for steps |
| **@step** | Decorator that registers async functions as steps |
| **Events** | Custom `Event` subclasses carry data between steps |
| **StartEvent** | Trigger that kicks off the workflow |
| **StopEvent** | Terminal event that returns the result |
| **Event routing** | Input type → step → output type (automatic dispatch) |

## Workflow Architecture
```
User Query (StartEvent)
    │
    ├─ parse_query()    →  QueryEvent(query, task_type)
    │
    ├─ retrieve_context()  →  RetrieveEvent(query, context, sources)
    │
    ├─ analyze()        →  AnalyzeEvent(analysis, sources)
    │
    └─ format_response()  →  StopEvent(result)
                                 ↓
                          Final Answer + Metadata
```

## Why Workflows?
- **Composable**: Each step is independent and testable
- **Observable**: Built-in event tracing shows execution flow
- **Branching**: Steps can emit different events based on conditions
- **Concurrent**: Multiple steps can run in parallel with proper event design
- **Production-ready**: Timeouts, error handling, and state management built-in